# BirdCLEF2026 EffNet Multiwindow — Fold0 Inference

- Model: EfficientNetV2-B0, single fold (fold0, ep16, val AUC 0.9794)
- Checkpoint: `ramkang/birdclef2026-effnet-multiwindow-fold0/best_fold0.pth`
- Mel: SR=32000, n_fft=1024, hop=320, n_mels=128, fmin=50, fmax=14000, top_db=80, XCL norm

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import timm
import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

In [ ]:
# ── Config ──
_BASE1 = Path('/kaggle/input/competitions/birdclef-2026')
_BASE2 = Path('/kaggle/input/birdclef-2026')
BASE = _BASE1 if _BASE1.exists() else _BASE2
print(f'BASE: {BASE}')

# dataset path: /kaggle/input/datasets/{user}/{slug}/
_MODEL1 = Path('/kaggle/input/datasets/ramkang/birdclef2026-effnet-multiwindow-fold0/best_fold0.pth')
_MODEL2 = Path('/kaggle/input/birdclef2026-effnet-multiwindow-fold0/best_fold0.pth')
MODEL_PATH = _MODEL1 if _MODEL1.exists() else _MODEL2
print(f'MODEL_PATH: {MODEL_PATH} (exists={MODEL_PATH.exists()})')

SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC  # 160000
FILE_SAMPLES = 60 * SR            # 1920000 (60s files)
N_WINDOWS = 12                    # 60s / 5s

N_FFT = 1024
HOP_LENGTH = 320
N_MELS = 128
F_MIN = 50.0
F_MAX = 14000.0
TOP_DB = 80.0
NORM_MEAN = -4.268
NORM_STD = 4.569

EFFNET_DIM = 1280
BATCH_SIZE = 32

# P100 (sm_60) 은 현재 PyTorch CUDA와 호환 안됨 → CPU
DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Labels ──
sample_sub = pd.read_csv(BASE / 'sample_submission.csv')
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
print(f'N_CLASSES: {N_CLASSES}')

In [ ]:
# ── Mel spectrogram ──
_mel_module = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=F_MIN,
    f_max=F_MAX,
)

def wav_to_spec(wav: np.ndarray) -> torch.Tensor:
    """wav (float32, WINDOW_SAMPLES) -> (1, 128, T) normalized spec."""
    x = torch.from_numpy(wav).unsqueeze(0)
    mel = _mel_module(x)
    db = 10.0 * torch.log10(mel.clamp(min=1e-10))
    max_val = db.flatten(-2).max(dim=-1).values[..., None, None]
    db = torch.maximum(db, max_val - TOP_DB)
    db = (db - NORM_MEAN) / NORM_STD
    return db

In [ ]:
# ── Model ──
class EffNetInfer(nn.Module):
    def __init__(self, n_classes, backbone='tf_efficientnetv2_b0', effnet_dim=EFFNET_DIM):
        super().__init__()
        self.stem_conv = nn.Conv2d(1, 3, kernel_size=3, stride=1, padding=1, bias=False)
        self.backbone = timm.create_model(backbone, pretrained=False,
                                          in_chans=3, num_classes=0)
        self.head = nn.Linear(effnet_dim, n_classes)

    def forward(self, x):
        x = self.stem_conv(x)
        feat = self.backbone(x)
        return self.head(feat)


model = EffNetInfer(n_classes=N_CLASSES).to(DEVICE)
state = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
infer_keys = {k: v for k, v in state.items()
              if k.startswith('stem_conv') or k.startswith('backbone') or k.startswith('head')}
model.load_state_dict(infer_keys, strict=True)
model.eval()
print(f'Model loaded: {len(infer_keys)} keys')

In [ ]:
# ── Dataset ──
class SoundscapeDataset(Dataset):
    def __init__(self, paths):
        self.items = []
        for p in tqdm(paths, desc='Loading audio'):
            stem = p.stem
            wav, sr = sf.read(str(p), dtype='float32', always_2d=False)
            if wav.ndim == 2:
                wav = wav.mean(axis=1)
            if len(wav) < FILE_SAMPLES:
                wav = np.pad(wav, (0, FILE_SAMPLES - len(wav)))
            else:
                wav = wav[:FILE_SAMPLES]
            for i in range(N_WINDOWS):
                start = i * WINDOW_SAMPLES
                chunk = wav[start: start + WINDOW_SAMPLES]
                row_id = f'{stem}_{(i + 1) * WINDOW_SEC}'
                self.items.append((chunk, row_id))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        chunk, row_id = self.items[idx]
        return wav_to_spec(chunk), row_id

print('SoundscapeDataset defined')

In [ ]:
# ── Inference ──
test_paths = sorted((BASE / 'test_soundscapes').glob('*.ogg'))
IS_DRY_RUN = (len(test_paths) == 0)

if IS_DRY_RUN:
    print('No hidden test — dry-run: zero-pred submission from sample_submission.csv')
    _dry = pd.read_csv(BASE / 'sample_submission.csv')
    all_row_ids = _dry['row_id'].tolist()
    all_probs = np.zeros((len(_dry), N_CLASSES), dtype=np.float32)
    print(f'Dry-run predictions shape: {all_probs.shape}')
else:
    print(f'Test files: {len(test_paths)}')
    ds = SoundscapeDataset(test_paths)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=False)
    all_row_ids = []
    all_probs = []
    with torch.no_grad():
        for specs, row_ids in tqdm(loader, desc='Inference'):
            logits = model(specs.to(DEVICE))
            probs = torch.sigmoid(logits).numpy().astype(np.float32)
            all_row_ids.extend(row_ids)
            all_probs.append(probs)
    all_probs = np.concatenate(all_probs, axis=0)
    print(f'Predictions shape: {all_probs.shape}')

In [ ]:
# ── Submission ──
submission = pd.DataFrame(all_probs, columns=PRIMARY_LABELS)
submission.insert(0, 'row_id', all_row_ids)

if not IS_DRY_RUN:
    assert len(submission) == len(test_paths) * N_WINDOWS, \
        f'Expected {len(test_paths) * N_WINDOWS} rows, got {len(submission)}'
assert not submission.isna().any().any()

submission.to_csv('submission.csv', index=False)
print(f'Saved submission.csv: {submission.shape}')
print(submission.head(3))